# 6. Push to GitHub

Syncs this whole project folder to a private GitHub repo. Re-run this
notebook top to bottom any time you want to push your latest changes —
it's idempotent: it only initializes the repo / creates the remote repo
once, and only commits when something actually changed.

**What gets pushed**: all notebooks, `aoi_export.py`, `requirements.txt`,
and the small stuff in `output/` (GeoJSON/KML/CSV). **What does NOT get
pushed**: the `downloads/` folder — it holds actual satellite data
(tens of GB, including files far bigger than GitHub's 100 MB per-file
limit). Instead, a `downloads_manifest.txt` listing every downloaded
file's name and size (no data) is generated and committed, so the repo
still has a record of what's been downloaded.

**Authentication**: GitHub removed password-based `git push` support in
2021 — you need a **Personal Access Token** instead, not your GitHub
account password. Create one (classic, `repo` scope is enough) at
https://github.com/settings/tokens, then either:
- set it once as a system environment variable named `GITHUB_TOKEN`, or
- leave it unset and this notebook will prompt for it via `getpass` each
  run (hidden input, never written to disk or into this notebook file).

**Requires**: `GITHUB_USERNAME` and `REPO_NAME` filled in below. The repo
is created automatically (as **private**) the first time you run this if
it doesn't already exist.

📖 Getting a 401 error, or need help creating a token? See
`docs/pdf/06_GitHub_Sync_Guide.pdf`.

In [21]:
GITHUB_USERNAME = "SAY70"  # <-- detected from your git config; change if this isn't your GitHub username
REPO_NAME = "satellite-data-search"  # <-- fill in (created automatically if it doesn't exist)
PRIVATE = True
BRANCH = "main"
COMMIT_MESSAGE = None  # <-- e.g. "Add NISAR support"; leave None to auto-generate a timestamped message

## Authenticate

Reads `GITHUB_TOKEN` from the environment if you've set it; otherwise
prompts for it (hidden input). The token is only ever held in memory for
this run — it's never written to `.git/config` or any file.

In [22]:
import os
from getpass import getpass

GITHUB_TOKEN = (os.environ.get("GITHUB_TOKEN") or getpass("GitHub Personal Access Token (hidden): ")).strip()
if not GITHUB_TOKEN:
    raise RuntimeError("No token provided — create one at https://github.com/settings/tokens and re-run.")
print("Token loaded (from environment)." if os.environ.get("GITHUB_TOKEN") else "Token loaded (entered now).")

Token loaded (entered now).


## Record what's in `downloads/` without pushing the actual data

Writes `downloads_manifest.txt` — every downloaded file's relative path
and size, nothing else. The real files stay out of git entirely (see
`.gitignore` below); this manifest is what actually gets committed.

In [23]:
from pathlib import Path

DOWNLOAD_DIR = Path("downloads")
manifest_path = Path("downloads_manifest.txt")

lines = []
if DOWNLOAD_DIR.exists():
    for f in sorted(DOWNLOAD_DIR.rglob("*")):
        if f.is_file():
            size_mb = f.stat().st_size / (1024 ** 2)
            lines.append(f"{f.relative_to(DOWNLOAD_DIR).as_posix()}\t{size_mb:,.1f} MB")

manifest_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
print(f"{len(lines)} downloaded file name(s) recorded in {manifest_path} (actual data files are not pushed).")

0 downloaded file name(s) recorded in downloads_manifest.txt (actual data files are not pushed).


## Make sure `.gitignore` excludes the data folder

In [24]:
gitignore_path = Path(".gitignore")
required_lines = ["downloads/", "__pycache__/", "*.pyc", ".ipynb_checkpoints/", "*.bak"]

existing = gitignore_path.read_text(encoding="utf-8").splitlines() if gitignore_path.exists() else []
missing = [line for line in required_lines if line not in existing]

if missing:
    with open(gitignore_path, "a", encoding="utf-8") as f:
        if existing and existing[-1] != "":
            f.write("\n")
        f.write("\n".join(missing) + "\n")
    print(f"Added to .gitignore: {missing}")
else:
    print(".gitignore already up to date.")

.gitignore already up to date.


## Initialize the local git repo (first run only)

`run()` is used for every git command below. If a command fails, its
error message has the token scrubbed before being shown or raised, so
the token can never end up printed into a saved cell output.

In [25]:
import subprocess


def run(cmd, redact=None):
    """Run a git command; scrub `redact` (if given) out of any output before it can surface."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    out, err = result.stdout, result.stderr
    if redact:
        out = out.replace(redact, "***")
        err = err.replace(redact, "***")
    if result.returncode != 0:
        raise RuntimeError(f"git command failed (exit {result.returncode}):\n{err or out}")
    return out.strip()


if not Path(".git").exists():
    run(["git", "init"])
    run(["git", "branch", "-M", BRANCH])
    print("Initialized new git repository.")
else:
    print("Git repository already initialized.")

Git repository already initialized.


## Create the GitHub repo if it doesn't exist yet

Uses the GitHub REST API directly (no `gh` CLI needed). Checks first —
if the repo already exists this is a no-op.

In [26]:
import requests

api_headers = {"Authorization": f"Bearer {GITHUB_TOKEN}", "Accept": "application/vnd.github+json"}
repo_api_url = f"https://api.github.com/repos/{GITHUB_USERNAME}/{REPO_NAME}"

resp = requests.get(repo_api_url, headers=api_headers, timeout=30)
if resp.status_code == 200:
    print(f"Repo {GITHUB_USERNAME}/{REPO_NAME} already exists (private={resp.json()['private']}).")
elif resp.status_code == 404:
    create_resp = requests.post(
        "https://api.github.com/user/repos",
        headers=api_headers,
        json={"name": REPO_NAME, "private": PRIVATE},
        timeout=30,
    )
    create_resp.raise_for_status()
    print(f"Created new {'private' if PRIVATE else 'public'} repo: {create_resp.json()['html_url']}")
else:
    resp.raise_for_status()

Repo SAY70/satellite-data-search already exists (private=True).


## Stage, commit (only if something changed), and push

The token is only ever passed as part of the push URL argument to a
single `git push` subprocess call — it's never saved into `.git/config`
(the `origin` remote is kept token-free) and never printed.

In [27]:
from datetime import datetime

run(["git", "add", "-A"])
status = run(["git", "status", "--porcelain"])

if status:
    message = COMMIT_MESSAGE or f"Update {datetime.now().strftime('%Y-%m-%d %H:%M')}"
    run(["git", "commit", "-m", message])
    print(f"Committed: {message}")
else:
    print("No changes to commit.")

remote_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
if "origin" not in run(["git", "remote"]).splitlines():
    run(["git", "remote", "add", "origin", remote_url])

# Token is passed only as an argument to this one push call, never saved to
# .git/config (the "origin" remote above stays token-free).
push_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
run(["git", "push", push_url, f"HEAD:{BRANCH}"], redact=GITHUB_TOKEN)

print(f"Pushed to https://github.com/{GITHUB_USERNAME}/{REPO_NAME} ({BRANCH})")

Committed: Update 2026-09-17 18:40


RuntimeError: git command failed (exit 1):
To https://github.com/SAY70/satellite-data-search.git
 ! [rejected]        HEAD -> main (non-fast-forward)
error: failed to push some refs to 'https://github.com/SAY70/satellite-data-search.git'
hint: Updates were rejected because the tip of your current branch is behind
hint: its remote counterpart. If you want to integrate the remote changes,
hint: use 'git pull' before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.
